# Build project_domain_map.json

This notebook reads the TCGA project universe from frozen source releases created by `_1_build_source_release.py`:

```text
eval/eval_datasets/TCGA_eval_data/source_releases/{tcga_source_id}/inventory/
```

It builds a `_3_init_benchmark.py`-compatible `project_domain_map.json`. `_3_init_benchmark.py` expects labels to be exactly one of:

```text
in_domain
out_of_domain
excluded
```


## Helpers

Run this section once. It defines all reusable functions used by every source-specific section.


In [1]:
from dataclasses import dataclass
from datetime import date
from pathlib import Path
import json
import sys
import pandas as pd
from IPython.display import display

# Add the project root to sys.path so that 'src' can be imported
project_root = Path.cwd().parents[3]
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.utils.constants import get_full_path


ALLOWED_LABELS = {"in_domain", "out_of_domain", "excluded"}

INVENTORY_FILES_IN_PRIORITY_ORDER = [
    "project_inventory.parquet",
    "selected_rna_files.parquet",
    "candidate_rna_files.parquet",
    "candidate_slide_files.parquet",
]


@dataclass(frozen=True)
class SourceReleaseContext:
    """Resolved paths for one TCGA source release."""

    tcga_source_id: str
    source_root: Path
    inventory_dir: Path
    settings_dir: Path

DATA_ROOT = get_full_path("eval", "eval_datasets", "TCGA_eval_data")
SOURCE_RELEASES_ROOT = DATA_ROOT / "source_releases"

print("TCGA data root:", DATA_ROOT)
print("Source releases root:", SOURCE_RELEASES_ROOT)


def resolve_source_release(
    tcga_source_id: str | None,
    source_releases_root: Path = SOURCE_RELEASES_ROOT,
) -> SourceReleaseContext:
    """Resolve paths for one source release.

    If `tcga_source_id` is None, the most recently modified source-release
    directory is used.
    """
    if not source_releases_root.exists():
        raise FileNotFoundError(source_releases_root)

    source_release_dirs = sorted(p for p in source_releases_root.iterdir() if p.is_dir())
    if not source_release_dirs:
        raise RuntimeError(f"No source release folders found under {source_releases_root}")

    if tcga_source_id is None:
        latest_source_dir = max(source_release_dirs, key=lambda p: p.stat().st_mtime)
        tcga_source_id = latest_source_dir.name
        print(
            "TCGA_SOURCE_ID was None, using most recently modified source release: "
            f"{tcga_source_id}"
        )
    else:
        print(f"Using explicit TCGA_SOURCE_ID: {tcga_source_id}")

    source_root = source_releases_root / tcga_source_id
    inventory_dir = source_root / "inventory"
    settings_dir = source_root / "settings"

    if not source_root.exists():
        raise FileNotFoundError(source_root)
    if not inventory_dir.exists():
        raise FileNotFoundError(inventory_dir)

    print("Source root:", source_root)
    print("Inventory dir:", inventory_dir)
    print("Settings dir:", settings_dir)

    return SourceReleaseContext(
        tcga_source_id=tcga_source_id,
        source_root=source_root,
        inventory_dir=inventory_dir,
        settings_dir=settings_dir,
    )


def read_project_ids_from_inventory(path: Path) -> set[str]:
    """Read unique `project_id` values from a parquet/csv/tsv inventory file."""
    if not path.exists():
        return set()

    if path.suffix.lower() == ".parquet":
        try:
            df = pd.read_parquet(path, columns=["project_id"])
        except Exception:
            df = pd.read_parquet(path)
    elif path.suffix.lower() == ".csv":
        df = pd.read_csv(path, dtype=str)
    elif path.suffix.lower() in {".tsv", ".txt"}:
        df = pd.read_csv(path, sep="\t", dtype=str)
    else:
        raise ValueError(f"Unsupported inventory file type: {path}")

    if "project_id" not in df.columns:
        return set()

    return {
        str(x).strip()
        for x in df["project_id"].dropna().astype(str)
        if str(x).strip()
    }


def collect_project_universe(
    context: SourceReleaseContext,
    inventory_files: list[str] | None = None,
) -> tuple[list[str], dict[str, list[str]]]:
    """Collect all TCGA project IDs from the frozen inventory files for one release."""
    inventory_files = inventory_files or INVENTORY_FILES_IN_PRIORITY_ORDER

    project_sources: dict[str, list[str]] = {}
    project_universe: set[str] = set()

    for name in inventory_files:
        path = context.inventory_dir / name
        ids = read_project_ids_from_inventory(path)
        if ids:
            project_sources[name] = sorted(ids)
            project_universe.update(ids)

    if not project_universe:
        raise RuntimeError(
            "Could not infer any TCGA projects from the frozen source release inventories. "
            "Check that `_1_build_source_release.py` completed and wrote inventory parquet files."
        )

    project_ids = sorted(project_universe)

    print(
        f"Found {len(project_ids)} TCGA projects in frozen source release: "
        f"{context.tcga_source_id}"
    )
    for source_name, ids in project_sources.items():
        print(f"- {source_name}: {len(ids)} projects")

    return project_ids, project_sources


def load_project_inventory_summary(
    context: SourceReleaseContext,
    project_ids: list[str],
    display_table: bool = True,
) -> pd.DataFrame:
    """Load and optionally display the project inventory summary written by `_1`."""
    project_inventory_path = context.inventory_dir / "project_inventory.parquet"

    if project_inventory_path.exists():
        project_inventory_df = pd.read_parquet(project_inventory_path)
        project_inventory_df = project_inventory_df.sort_values("project_id").reset_index(drop=True)
    else:
        project_inventory_df = pd.DataFrame({"project_id": project_ids})

    if display_table:
        display(project_inventory_df)

    return project_inventory_df


def build_project_domain_map(
    project_ids: list[str],
    default_label: str = "in_domain",
    project_label_overrides: dict[str, str] | None = None,
) -> tuple[dict[str, str], pd.DataFrame]:
    """Build the project -> label map; overrides outside this benchmark's source release are ignored."""
    project_label_overrides = project_label_overrides or {}

    if default_label not in ALLOWED_LABELS:
        raise ValueError(
            f"DEFAULT_LABEL must be one of {sorted(ALLOWED_LABELS)}, got {default_label!r}"
        )

    bad_override_labels = {
        project: label
        for project, label in project_label_overrides.items()
        if label not in ALLOWED_LABELS
    }
    if bad_override_labels:
        raise ValueError(f"Invalid labels in project_label_overrides: {bad_override_labels}")

    unknown_override_projects = sorted(set(project_label_overrides) - set(project_ids))
    if unknown_override_projects:
        print(f"Info: project_label_overrides contains projects not in this benchmark's source release; ignoring them: {unknown_override_projects}")

    project_domain_map = {project_id: default_label for project_id in project_ids}
    project_domain_map.update({project: label for project, label in project_label_overrides.items() if project in set(project_ids)})

    mapping_df = (
        pd.DataFrame(
            {
                "project_id": list(project_domain_map.keys()),
                "label": list(project_domain_map.values()),
            }
        )
        .sort_values("project_id")
        .reset_index(drop=True)
    )

    return project_domain_map, mapping_df


def display_project_domain_map(mapping_df: pd.DataFrame) -> None:
    """Display the full mapping table and a label-count summary."""
    display(mapping_df)
    display(mapping_df["label"].value_counts().rename_axis("label").reset_index(name="n_projects"))


def build_project_domain_map_json(
    context: SourceReleaseContext,
    project_domain_map: dict[str, str],
    reviewed: bool,
    reviewer: str,
    review_date: str | None = None,
    notes: str | None = None,
    preview_chars: int = 4000,
) -> dict:
    """Build the JSON object written to project_domain_map.template.json/json."""
    review_date = review_date or date.today().isoformat()
    notes = notes or (
        "Project universe was derived from frozen source release inventories "
        "written by _1_build_source_release.py. Labels were manually reviewed "
        "for _3_init_benchmark.py."
    )

    obj = {
        "manual_review": {
            "reviewed": reviewed,
            "reviewer": reviewer,
            "review_date": review_date,
            "allowed_labels": sorted(ALLOWED_LABELS),
            "notes": notes,
        },
        "tcga_source_id": context.tcga_source_id,
        "source_root": str(context.source_root),
        "project_domain_map": project_domain_map,
    }

    preview = json.dumps(obj, indent=2, sort_keys=True)
    print(preview[:preview_chars])
    if len(preview) > preview_chars:
        print("\n... truncated preview ...")

    return obj


def save_project_domain_map_template(
    context: SourceReleaseContext,
    obj: dict,
    filename: str = "project_domain_map.template.json",
) -> Path:
    """Save a non-final template copy next to the source-release settings."""
    context.settings_dir.mkdir(parents=True, exist_ok=True)
    template_path = context.settings_dir / filename

    with template_path.open("w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, sort_keys=True)
        f.write("\n")

    print("Saved template:", template_path)
    return template_path


def validate_project_domain_map_for_init_benchmark(
    obj: dict,
    expected_projects: set[str] | list[str],
) -> None:
    """Validate the checks expected by `_3_init_benchmark.py`."""
    expected_projects = {str(x) for x in expected_projects}

    meta = obj.get("manual_review") or obj.get("metadata") or {}
    if not bool(meta.get("reviewed")):
        raise RuntimeError("Set REVIEWED = True only after manual review, then rerun the JSON/save cells.")

    for field in ["reviewer", "review_date", "allowed_labels"]:
        if field not in meta:
            raise RuntimeError(f"Missing manual-review metadata field: {field}")

    allowed = set(meta.get("allowed_labels") or [])
    required = {"in_domain", "out_of_domain", "excluded"}
    if allowed != required:
        raise RuntimeError(f"allowed_labels must be exactly {sorted(required)}")

    labels = obj.get("project_domain_map") or obj.get("projects") or {}
    if not isinstance(labels, dict) or not labels:
        raise RuntimeError("project_domain_map has no project labels")

    bad = {k: v for k, v in labels.items() if v not in required}
    if bad:
        raise RuntimeError(f"Invalid project labels: {bad}")

    labeled = {str(x) for x in labels.keys()}
    missing = sorted(expected_projects - labeled)
    extra = sorted(labeled - expected_projects)

    if missing:
        raise RuntimeError(f"Missing labels for frozen source projects: {missing}")
    if extra:
        raise RuntimeError(f"Map contains projects absent from frozen source release: {extra}")


def save_project_domain_map_final(
    context: SourceReleaseContext,
    obj: dict,
    expected_projects: set[str] | list[str],
    filename: str = "project_domain_map.json",
) -> Path:
    """Validate and save the final `project_domain_map.json`."""
    validate_project_domain_map_for_init_benchmark(obj, expected_projects)

    context.settings_dir.mkdir(parents=True, exist_ok=True)
    output_path = context.settings_dir / filename
    tmp_path = output_path.with_suffix(output_path.suffix + ".tmp")

    with tmp_path.open("w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, sort_keys=True)
        f.write("\n")

    tmp_path.replace(output_path)

    print("Saved final project_domain_map.json:")
    print(output_path)
    print()
    print("Use with _3_init_benchmark.py:")
    print(f"--project-domain-map {output_path}")

    return output_path


def reload_and_inspect_final_project_domain_map(
    context: SourceReleaseContext,
    filename: str = "project_domain_map.json",
    display_table: bool = True,
) -> dict | None:
    """Reload and optionally display the final JSON if it exists."""
    final_path = context.settings_dir / filename

    if not final_path.exists():
        print("Final JSON does not exist yet. Set REVIEWED=True, SAVE_FINAL=True, then rerun the save cell.")
        return None

    with final_path.open("r", encoding="utf-8") as f:
        reloaded = json.load(f)

    print("Reloaded:", final_path)
    print("reviewed:", reloaded.get("manual_review", {}).get("reviewed"))
    print("n projects:", len(reloaded.get("project_domain_map", {})))

    if display_table:
        display(
            pd.DataFrame(
                {
                    "project_id": list(reloaded["project_domain_map"].keys()),
                    "label": list(reloaded["project_domain_map"].values()),
                }
            )
            .sort_values("project_id")
            .reset_index(drop=True)
        )

    return reloaded


TCGA data root: /path/to/ALHENA_data/eval/eval_datasets/TCGA_eval_data
Source releases root: /path/to/ALHENA_data/eval/eval_datasets/TCGA_eval_data/source_releases


## tcga_benchmark_1

In [2]:
# Step 1 — Source-specific settings

BENCHMARK_ID = "tcga_benchmark_1"
TCGA_SOURCE_ID = "gdc_tcga_20260519_starcounts_dxslides_v1"
DATASET_ID = "ds_ct1_20x_224_o112"
TRAINING_DATASET_ID = "ds_ct1_20x_224_o112_1"

# Use only:
# - "in_domain"
# - "out_of_domain"
# - "excluded"
# Set everything to DEFAULT_LABEL by default, then manually change only the few projects that should be different using project_label_overrides.
DEFAULT_LABEL = "in_domain"

# Edit this dictionary manually for this TCGA_SOURCE_ID.
project_label_overrides = {
    # Not in the training tissue list
    "TCGA-ACC": "out_of_domain",   # Adrenal cortex
    "TCGA-BLCA": "out_of_domain",  # Bladder
    "TCGA-CHOL": "out_of_domain",  # Bile duct / cholangiocarcinoma, not LIHC liver
    "TCGA-ESCA": "out_of_domain",  # Esophagus
    "TCGA-GBM": "out_of_domain",   # Brain
    "TCGA-HNSC": "out_of_domain",  # Mixed head/neck sites; not safely equal to Tonsil at project level
    "TCGA-LGG": "out_of_domain",   # Brain
    "TCGA-MESO": "out_of_domain",  # Mesothelioma, usually pleura/peritoneum
    "TCGA-PCPG": "out_of_domain",  # Pheochromocytoma/paraganglioma
    "TCGA-SARC": "out_of_domain",  # Sarcoma, heterogeneous soft tissue/bone
    "TCGA-STAD": "out_of_domain",  # Stomach
    "TCGA-TGCT": "out_of_domain",  # Testis
    "TCGA-THCA": "out_of_domain",  # Thyroid
    "TCGA-THYM": "out_of_domain",  # Thymus
    "TCGA-UCEC": "out_of_domain",  # Uterus corpus/endometrium, not cervix
    "TCGA-UCS": "out_of_domain",   # Uterus carcinosarcoma, not cervix
    "TCGA-UVM": "out_of_domain",   # Uveal/eye melanoma
    # Excluded
    "TCGA-LAML": "excluded",       # Acute myeloid leukemia; blood/bone-marrow disease, not a listed solid tissue
}
# In-domain projects:
# Breast        -> TCGA-BRCA
# Lung          -> TCGA-LUAD, TCGA-LUSC
# Skin          -> TCGA-SKCM
# Pancreatic    -> TCGA-PAAD
# Colon         -> TCGA-COAD, and (TCGA-READ is accepted as colorectal)
# Kidney        -> TCGA-KICH, TCGA-KIRC, TCGA-KIRP
# Liver         -> TCGA-LIHC
# LymphNode     -> TCGA-DLBC
# Ovarian       -> TCGA-OV
# Prostate      -> TCGA-PRAD
# Cervix        -> TCGA-CESC


# Keep REVIEWED = False until the table below has been checked manually.
REVIEWED = True
REVIEWER = "reviewer_1"

# Template saving is safe before final manual review.
SAVE_TEMPLATE = False

# Set SAVE_FINAL = True only after setting REVIEWED = True.
SAVE_FINAL = True


In [3]:
# Step 2 — Resolve source-release paths
context = resolve_source_release(TCGA_SOURCE_ID)


Using explicit TCGA_SOURCE_ID: gdc_tcga_20260519_starcounts_dxslides_v1
Source root: /path/to/ALHENA_data/eval/eval_datasets/TCGA_eval_data/source_releases/gdc_tcga_20260519_starcounts_dxslides_v1
Inventory dir: /path/to/ALHENA_data/eval/eval_datasets/TCGA_eval_data/source_releases/gdc_tcga_20260519_starcounts_dxslides_v1/inventory
Settings dir: /path/to/ALHENA_data/eval/eval_datasets/TCGA_eval_data/source_releases/gdc_tcga_20260519_starcounts_dxslides_v1/settings


In [4]:
# Step 3 — Read TCGA projects from the frozen inventory files
project_ids, project_sources = collect_project_universe(context)


Found 10 TCGA projects in frozen source release: gdc_tcga_20260519_starcounts_dxslides_v1
- project_inventory.parquet: 10 projects
- selected_rna_files.parquet: 10 projects
- candidate_rna_files.parquet: 10 projects
- candidate_slide_files.parquet: 10 projects


In [5]:
# Step 4 — Display project inventory summary
project_inventory_df = load_project_inventory_summary(context, project_ids)


,project_id,n_candidate_rna_files,n_candidate_rna_cases,n_selected_rna_files_for_download,n_selected_rna_cases_for_download,n_candidate_slide_files,n_candidate_slide_cases,n_resolved_rna_parents,n_resolved_slide_parents
0,TCGA-BRCA,1231,1095,1079,1079,1113,1043,1231,1113
1,TCGA-CESC,309,304,302,302,275,265,309,275
2,TCGA-COAD,524,458,397,397,396,388,524,396
3,TCGA-KIRC,614,533,457,457,440,434,614,440
4,TCGA-LIHC,424,371,366,366,369,355,424,369
5,TCGA-LUAD,601,518,452,452,464,413,601,464
6,TCGA-OV,434,427,415,415,84,83,434,84
7,TCGA-PAAD,183,178,178,178,202,176,183,202
8,TCGA-PRAD,554,497,485,485,438,392,554,438
9,TCGA-SKCM,473,469,103,103,110,98,473,110


In [6]:
project_inventory_df.project_id

0    TCGA-BRCA
1    TCGA-CESC
2    TCGA-COAD
3    TCGA-KIRC
4    TCGA-LIHC
5    TCGA-LUAD
6      TCGA-OV
7    TCGA-PAAD
8    TCGA-PRAD
9    TCGA-SKCM
Name: project_id, dtype: str

In [7]:
# List of tissue for given training_dataset_id
info_training_dataset_id_path = get_full_path("train", "training_datasets", DATASET_ID, TRAINING_DATASET_ID, "informations", f"infos_{TRAINING_DATASET_ID}.csv")
info_training_dataset_id_df = pd.read_csv(info_training_dataset_id_path)
for t in info_training_dataset_id_df[info_training_dataset_id_df.set=="train"]["type"].unique():
    print(t)

Breast
Lung
Skin
Pancreatic
Heart
Colon
Kidney
Liver
Tonsil
LymphNode
Ovarian
Prostate
Cervix


In [8]:
# Step 5 — Build and display project labels
project_domain_map, mapping_df = build_project_domain_map(
    project_ids=project_ids,
    default_label=DEFAULT_LABEL,
    project_label_overrides=project_label_overrides,
)

display_project_domain_map(mapping_df)


Info: project_label_overrides contains projects not in this benchmark's source release; ignoring them: ['TCGA-ACC', 'TCGA-BLCA', 'TCGA-CHOL', 'TCGA-ESCA', 'TCGA-GBM', 'TCGA-HNSC', 'TCGA-LAML', 'TCGA-LGG', 'TCGA-MESO', 'TCGA-PCPG', 'TCGA-SARC', 'TCGA-STAD', 'TCGA-TGCT', 'TCGA-THCA', 'TCGA-THYM', 'TCGA-UCEC', 'TCGA-UCS', 'TCGA-UVM']


,project_id,label
0,TCGA-BRCA,in_domain
1,TCGA-CESC,in_domain
2,TCGA-COAD,in_domain
3,TCGA-KIRC,in_domain
4,TCGA-LIHC,in_domain
5,TCGA-LUAD,in_domain
6,TCGA-OV,in_domain
7,TCGA-PAAD,in_domain
8,TCGA-PRAD,in_domain
9,TCGA-SKCM,in_domain


,label,n_projects
0,in_domain,10


In [9]:
# Step 6 — Build the JSON object
project_domain_map_json = build_project_domain_map_json(
    context=context,
    project_domain_map=project_domain_map,
    reviewed=REVIEWED,
    reviewer=REVIEWER,
)
project_domain_map_json

{
  "manual_review": {
    "allowed_labels": [
      "excluded",
      "in_domain",
      "out_of_domain"
    ],
    "notes": "Project universe was derived from frozen source release inventories written by _1_build_source_release.py. Labels were manually reviewed for _3_init_benchmark.py.",
    "review_date": "2026-06-04",
    "reviewed": true,
    "reviewer": "reviewer_1"
  },
  "project_domain_map": {
    "TCGA-BRCA": "in_domain",
    "TCGA-CESC": "in_domain",
    "TCGA-COAD": "in_domain",
    "TCGA-KIRC": "in_domain",
    "TCGA-LIHC": "in_domain",
    "TCGA-LUAD": "in_domain",
    "TCGA-OV": "in_domain",
    "TCGA-PAAD": "in_domain",
    "TCGA-PRAD": "in_domain",
    "TCGA-SKCM": "in_domain"
  },
  "source_root": "/path/to/ALHENA_data/eval/eval_datasets/TCGA_eval_data/source_releases/gdc_tcga_20260519_starcounts_dxslides_v1",
  "tcga_source_id": "gdc_tcga_20260519_starcounts_dxslides_v1"
}


{'manual_review': {'reviewed': True,
  'reviewer': 'reviewer_1',
  'review_date': '2026-06-04',
  'allowed_labels': ['excluded', 'in_domain', 'out_of_domain'],
  'notes': 'Project universe was derived from frozen source release inventories written by _1_build_source_release.py. Labels were manually reviewed for _3_init_benchmark.py.'},
 'tcga_source_id': 'gdc_tcga_20260519_starcounts_dxslides_v1',
 'source_root': '/path/to/ALHENA_data/eval/eval_datasets/TCGA_eval_data/source_releases/gdc_tcga_20260519_starcounts_dxslides_v1',
 'project_domain_map': {'TCGA-BRCA': 'in_domain',
  'TCGA-CESC': 'in_domain',
  'TCGA-COAD': 'in_domain',
  'TCGA-KIRC': 'in_domain',
  'TCGA-LIHC': 'in_domain',
  'TCGA-LUAD': 'in_domain',
  'TCGA-OV': 'in_domain',
  'TCGA-PAAD': 'in_domain',
  'TCGA-PRAD': 'in_domain',
  'TCGA-SKCM': 'in_domain'}}

In [10]:
# Step 7 — Save a template copy
if SAVE_TEMPLATE:
    template_path = save_project_domain_map_template(context, project_domain_map_json)
else:
    print("Template save skipped because SAVE_TEMPLATE is False.")


Template save skipped because SAVE_TEMPLATE is False.


In [11]:
# Step 8 — Validate and save the final project_domain_map.json
if SAVE_FINAL:
    output_path = save_project_domain_map_final(
        context=context,
        obj=project_domain_map_json,
        expected_projects=project_ids,
    )
else:
    print("Final save skipped. After manual review, set REVIEWED=True and SAVE_FINAL=True, then rerun steps 6 and 8.")


Saved final project_domain_map.json:
/path/to/ALHENA_data/eval/eval_datasets/TCGA_eval_data/source_releases/gdc_tcga_20260519_starcounts_dxslides_v1/settings/project_domain_map.json

Use with _3_init_benchmark.py:
--project-domain-map /path/to/ALHENA_data/eval/eval_datasets/TCGA_eval_data/source_releases/gdc_tcga_20260519_starcounts_dxslides_v1/settings/project_domain_map.json


In [12]:
# Step 9 — Optional: reload and inspect the final JSON
reloaded_project_domain_map_json = reload_and_inspect_final_project_domain_map(context)


Reloaded: /path/to/ALHENA_data/eval/eval_datasets/TCGA_eval_data/source_releases/gdc_tcga_20260519_starcounts_dxslides_v1/settings/project_domain_map.json
reviewed: True
n projects: 10


,project_id,label
0,TCGA-BRCA,in_domain
1,TCGA-CESC,in_domain
2,TCGA-COAD,in_domain
3,TCGA-KIRC,in_domain
4,TCGA-LIHC,in_domain
5,TCGA-LUAD,in_domain
6,TCGA-OV,in_domain
7,TCGA-PAAD,in_domain
8,TCGA-PRAD,in_domain
9,TCGA-SKCM,in_domain
